In [14]:
# 📍 Google Maps Places API (New) - Austin Place ID Scraper
# ===========================================================
# This notebook scrapes ONLY place IDs (FREE tier) for various 
# categories in Austin, TX using the Text Search (New) API

# ## 1. Setup and Installation

# Run this cell to install required packages

import requests
import json
import time
import pandas as pd
import folium
from datetime import datetime
import ipywidgets as widgets
from IPython.display import display, HTML, clear_output
import base64
import csv
from io import StringIO
import matplotlib.pyplot as plt
import os
from dotenv import load_dotenv
from pathlib import Path

load_dotenv()

print("✅ All packages installed and imported successfully!")

✅ All packages installed and imported successfully!


In [15]:
# ## 2. Configuration and API Key Setup
# =====================================

# Enter your Google Maps API key below
# Get one from: https://console.cloud.google.com/

API_KEY = os.getenv('places_api_key')


In [23]:
# ## ⚡ Ultra-Simple Test with Location Bias
# ==========================================

import requests

# Austin coordinates (downtown)
AUSTIN_LAT = 30.339468805734047
AUSTIN_LNG = -97.71764056388106

# Simple test with location bias
url = "https://places.googleapis.com/v1/places:searchText"
headers = {
    'Content-Type': 'application/json',
    'X-Goog-Api-Key': API_KEY,
    'X-Goog-FieldMask': 'places.id'  # Only IDs = FREE
}

payload = {
    "textQuery": "coffee shop",
    "pageSize": 20,
    "locationBias": {
        "circle": {
            "center": {
                "latitude": AUSTIN_LAT,
                "longitude": AUSTIN_LNG
            },
            "radius": 500 # 5km radius around downtown
        }
    }
}

response = requests.post(url, headers=headers, json=payload)

if response.status_code == 200:
    data = response.json()
    places = data.get('places', [])
    print(f"✅ Found {len(places)} coffee shops in Austin!")
    print("\n📝 Place IDs:")
    for i, place in enumerate(places, 1):
        print(f"   {i:2d}. {place['id']}")
    
    # Show first place's ID for testing
    if places:
        print(f"\n🎯 First place ID: {places[0]['id']}")
else:
    print(f"❌ Error: {response.status_code}")
    print(response.text)

✅ Found 20 coffee shops in Austin!

📝 Place IDs:
    1. ChIJG-gJw2vKRIYROWi2uwOp8QE
    2. ChIJSdFa-_vLRIYRV3BPw9M2PRY
    3. ChIJ1d_fnx3LRIYRsXBuZB0_P6U
    4. ChIJfborXjbKRIYRmc72vAbWATc
    5. ChIJlaDbM4bLRIYRprannR6KQpU
    6. ChIJa7zfoiDLRIYR1Sb-iOrB62E
    7. ChIJPzLifSq1RIYRDr8YJLRgRXo
    8. ChIJK_G24QHrRIYRFsqJnuQaKv0
    9. ChIJP8p2kJC1RIYR2qrGoXZtZCk
   10. ChIJDbXf03G1RIYROhj9Cwwv600
   11. ChIJ4Xs1GXy1RIYRbBEAnYEmswc
   12. ChIJJzMxpDfKRIYRwl82WgWySn0
   13. ChIJmUlBPt7LRIYRlh29CDdsl34
   14. ChIJrSya_8_LRIYR-ksiOahed7c
   15. ChIJMX0PA7HLRIYRzLknsCCBHf8
   16. ChIJleSYLIrLRIYRsjwo0HYdv2Q
   17. ChIJlcsHDRbKRIYRRve-J-f_w5k
   18. ChIJzd8l9ffLRIYRqKj1OpnM0hU
   19. ChIJjQie2fjLRIYRippgYxvy5XY
   20. ChIJd6l9YGvLRIYREe_-6cLC9_I

🎯 First place ID: ChIJG-gJw2vKRIYROWi2uwOp8QE


In [25]:
url

'https://places.googleapis.com/v1/places:searchText'

In [24]:
# ## 🎯 SIMPLE TEST: One Place ID → One DataFrame Row (with Display Name)
# ========================================================================

import requests
import pandas as pd

def get_place_dataframe(place_id, api_key):
    """
    Get place details and return as a single-row DataFrame
    """
    
    url = f"https://places.googleapis.com/v1/places/{place_id}"
    
    # Added 'displayName' to the field mask
    headers = {
        'Content-Type': 'application/json',
        'X-Goog-Api-Key': api_key,
        'X-Goog-FieldMask': 'displayName,location,formattedAddress,types'
    }
    
    print(f"🔍 Fetching: {place_id[:20]}...")
    
    response = requests.get(url, headers=headers)
    
    if response.status_code == 200:
        data = response.json()
        
        # Extract display name (it's nested in the response)
        display_name = data.get('displayName', {}).get('text', 'N/A')
        
        # Create a simple dataframe with one row - added 'name' column
        df = pd.DataFrame([{
            'place_id': place_id,
            'name': display_name,  # New column!
            'address': data.get('formattedAddress', 'N/A'),
            'lat': data.get('location', {}).get('latitude', None),
            'lng': data.get('location', {}).get('longitude', None),
            'types': ', '.join(data.get('types', []))  # Convert list to string
        }])
        
        return df
    else:
        print(f"❌ Error: {response.status_code}")
        print(response.text)
        return None

# --- JUST RUN THIS ---

my_place_id = "ChIJG-gJw2vKRIYROWi2uwOp8QE"  # Your place ID

# Get the dataframe
df = get_place_dataframe(my_place_id, API_KEY)

# Display it
if df is not None:
    print("\n✅ Here's your dataframe with store name:")
    display(df)

🔍 Fetching: ChIJG-gJw2vKRIYROWi2...

✅ Here's your dataframe with store name:


,place_id,name,address,lat,lng,types
0,ChIJG-gJw2vKRIYROWi2uwOp8QE,Epoch Coffee,"221 W N Loop Blvd, Austin, TX 78751, USA",30.318604,-97.72454,"coffee_shop, cafe, food_store, food, store, po..."


In [19]:
print(data)

{'places': [{'id': 'ChIJG-gJw2vKRIYROWi2uwOp8QE'}, {'id': 'ChIJK_G24QHrRIYRFsqJnuQaKv0'}, {'id': 'ChIJeR5RmGC1RIYRKkkzoEoDs-0'}, {'id': 'ChIJP8p2kJC1RIYR2qrGoXZtZCk'}, {'id': 'ChIJ_YuZ6prPRIYRqSepJcYkCrc'}, {'id': 'ChIJA7CVXZW0RIYRVSZIlvlwJbA'}, {'id': 'ChIJ2b80N5TNRIYRz75PknHy5B8'}, {'id': 'ChIJ9WFmrFe1RIYRJ_X20Y0Qr2s'}, {'id': 'ChIJSdFa-_vLRIYRV3BPw9M2PRY'}, {'id': 'ChIJDbXf03G1RIYROhj9Cwwv600'}, {'id': 'ChIJTzmFkkW0RIYRZhkrPHixqNA'}, {'id': 'ChIJ7agecQG1RIYRmhih01aU_-A'}, {'id': 'ChIJlaDbM4bLRIYRprannR6KQpU'}, {'id': 'ChIJa7zfoiDLRIYR1Sb-iOrB62E'}, {'id': 'ChIJkTM4la61RIYRm51Cg7lRuLg'}, {'id': 'ChIJK3Z4pP7NRIYR8O7wUssUOUo'}, {'id': 'ChIJf4kFMPu0RIYR96Fyje5TjLs'}, {'id': 'ChIJMX0PA7HLRIYRzLknsCCBHf8'}, {'id': 'ChIJ4yWPfKe1RIYRaY1w7rNsFIc'}, {'id': 'ChIJ6ay9LCzNRIYRa0oQX73Nqio'}]}
